# 11. All Category Topic Generation + Sample Classification

전체 eligible category/sentiment group을 대상으로 다음 단계를 실행합니다.

1. 그룹별 샘플링
2. Rule Profile 생성
3. Topic Pool 생성
4. Topic Pool 기준 샘플 memo 분류
5. `classification_detail` 저장

주의: 전체 카테고리는 LLM 호출량이 많습니다. 먼저 대상 그룹 수를 확인한 뒤 실행하세요.

In [ ]:
%sh
cd /Workspace/Users/jungryo.lee@lge.com/prj_TV_voc && git pull

In [ ]:
import sys
import time
import traceback
import importlib

from pyspark.sql import functions as F

PROJECT_ROOT = "/Workspace/Users/jungryo.lee@lge.com/prj_TV_voc"
SRC_ROOT = f"{PROJECT_ROOT}/src"

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)

import common.config_loader as config_loader
import pipeline.run_taxonomy_design_refresh as design_refresh
import pipeline.run_taxonomy_classification_batch as batch_runner

importlib.reload(config_loader)
importlib.reload(design_refresh)
importlib.reload(batch_runner)

from common.config_loader import get_log_table, get_output_table, load_config
from pipeline.run_taxonomy_design_refresh import load_design_target_groups
from pipeline.run_taxonomy_classification_batch import run_taxonomy_classification_batch

config = load_config(f"{PROJECT_ROOT}/config/settings.yaml")

RULE_PROFILE_TABLE = get_output_table(config, "rule_profile")
TOPIC_POOL_TABLE = get_output_table(config, "topic_pool")
CLASSIFICATION_DETAIL_TABLE = get_output_table(config, "classification_detail")
PIPELINE_PROGRESS_TABLE = get_log_table(config, "pipeline_progress")

print("config loaded")

In [ ]:
# 전체 카테고리 실행 설정
MODEL_KEY = "gpt_55"

# None이면 전체 eligible group 대상입니다.
TARGET_CATE_1_DEPTH = None
TARGET_CATE_2_DEPTH = None
TARGET_SC_MEASUREMENT = None

# 전체 실행 전 빠른 검증이 필요하면 숫자를 넣으세요. 전체 실행은 None.
LIMIT_GROUP_COUNT = None

# 주제별 분포 확인용 샘플 분류 건수입니다. 운영 전수 분류가 아니라 샘플 검증 단계입니다.
MAX_ROWS_PER_GROUP = 100

# True이면 대상 그룹만 보여주고 실제 LLM 실행은 하지 않습니다.
DRY_RUN_GROUPS_ONLY = True

# 일시적 LLM/API/클러스터 오류가 날 때 전체 batch를 자동 재시도합니다.
# resume_from_checkpoint=True라서 성공 완료된 그룹은 다음 시도에서 건너뜁니다.
AUTO_RETRY_RUN = True
MAX_RUN_ATTEMPTS = 5
RETRY_WAIT_SECONDS = 60

MODEL_VERSION_VALUE = config["llm"]["models"][MODEL_KEY]["model_version"]
PROMPT_VERSION = config["version"]["prompt_version"]
TAXONOMY_VERSION = config["version"]["taxonomy_version"]

print({
    "model_key": MODEL_KEY,
    "model_version": MODEL_VERSION_VALUE,
    "target_cate_1_depth": TARGET_CATE_1_DEPTH or "*",
    "target_cate_2_depth": TARGET_CATE_2_DEPTH or "*",
    "target_sc_measurement": TARGET_SC_MEASUREMENT if TARGET_SC_MEASUREMENT is not None else "*",
    "limit_group_count": LIMIT_GROUP_COUNT,
    "max_rows_per_group": MAX_ROWS_PER_GROUP,
    "dry_run_groups_only": DRY_RUN_GROUPS_ONLY,
    "auto_retry_run": AUTO_RETRY_RUN,
    "max_run_attempts": MAX_RUN_ATTEMPTS,
    "retry_wait_seconds": RETRY_WAIT_SECONDS,
})

In [ ]:
target_groups = load_design_target_groups(
    spark=spark,
    config=config,
    limit_group_count=LIMIT_GROUP_COUNT,
    cate_1_depth=TARGET_CATE_1_DEPTH,
    cate_2_depth=TARGET_CATE_2_DEPTH,
    sc_measurement=TARGET_SC_MEASUREMENT,
)

print({"target_group_count": len(target_groups)})

if target_groups:
    display(
        spark.createDataFrame(target_groups)
        .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement")
    )
else:
    print("No target groups found. Check source table, min_group_size, and target filters.")

In [ ]:
if DRY_RUN_GROUPS_ONLY:
    raise SystemExit("DRY_RUN_GROUPS_ONLY=True: 대상 그룹 확인 후 False로 바꾸고 다음 셀부터 다시 실행하세요.")

In [ ]:
last_error = None
attempt_count = MAX_RUN_ATTEMPTS if AUTO_RETRY_RUN else 1

for attempt in range(1, attempt_count + 1):
    try:
        print(f"[all_category_sample] attempt {attempt}/{attempt_count}")
        batch_result = run_taxonomy_classification_batch(
            spark,
            config=config,
            model_key=MODEL_KEY,
            cate_1_depth=TARGET_CATE_1_DEPTH,
            cate_2_depth=TARGET_CATE_2_DEPTH,
            sc_measurement=TARGET_SC_MEASUREMENT,
            limit_group_count=LIMIT_GROUP_COUNT,
            max_rows_per_group=MAX_ROWS_PER_GROUP,
            use_llm_fallback=True,
            resume_from_checkpoint=True,
            cleanup_checkpoint_on_success=True,
            print_progress=True,
        )
        last_error = None
        break
    except Exception as error:
        last_error = error
        print(f"[all_category_sample] attempt failed {attempt}/{attempt_count}: {error}")
        traceback.print_exc()
        if attempt >= attempt_count:
            break
        print(f"[all_category_sample] retry after {RETRY_WAIT_SECONDS} seconds")
        time.sleep(RETRY_WAIT_SECONDS)

if last_error is not None:
    raise last_error


print({
    "group_count": batch_result["group_count"],
    "processed_group_count": batch_result["processed_group_count"],
    "skipped_group_count": batch_result["skipped_group_count"],
    "classification_count": batch_result["classification_count"],
    "total_row_count": batch_result["total_row_count"],
    "total_overall_count": batch_result["total_overall_count"],
    "total_topic_count": batch_result["total_topic_count"],
    "total_others_count": batch_result["total_others_count"],
    "total_llm_used_count": batch_result["total_llm_used_count"],
    "checkpoint_key": batch_result["checkpoint_key"],
    "saved_tables": batch_result["saved_tables"],
})

In [ ]:
if batch_result["classification_summaries"]:
    display(
        spark.createDataFrame(batch_result["classification_summaries"])
        .orderBy(F.col("others_ratio").desc(), F.col("row_count").desc())
    )
else:
    print("No classification summaries returned.")

In [ ]:
detail_df = (
    spark.table(CLASSIFICATION_DETAIL_TABLE)
    .where(F.col("model_version") == MODEL_VERSION_VALUE)
    .where(F.col("prompt_version") == PROMPT_VERSION)
    .where(F.col("taxonomy_version") == TAXONOMY_VERSION)
)

if TARGET_CATE_1_DEPTH is not None:
    detail_df = detail_df.where(F.col("cate_1_depth") == TARGET_CATE_1_DEPTH)
if TARGET_CATE_2_DEPTH is not None:
    detail_df = detail_df.where(F.col("cate_2_depth") == TARGET_CATE_2_DEPTH)
if TARGET_SC_MEASUREMENT is not None:
    detail_df = detail_df.where(F.col("sc_measurement") == TARGET_SC_MEASUREMENT)

display(
    detail_df.groupBy("cate_1_depth", "cate_2_depth", "sc_measurement", "pred_topic", "pred_topic_type")
    .agg(
        F.count("*").alias("memo_cnt"),
        F.countDistinct("memo_id").alias("distinct_memo_id_cnt"),
    )
    .orderBy("cate_1_depth", "cate_2_depth", "sc_measurement", F.col("memo_cnt").desc())
)

In [ ]:
display(
    detail_df.select(
        "cate_1_depth",
        "cate_2_depth",
        "sc_measurement",
        "memo",
        "pred_topic",
        "pred_topic_type",
        "classification_stage",
        "confidence_score",
        "match_reason",
    )
    .orderBy(F.rand(42))
    .limit(100)
)

In [ ]:
# 실패/중단 시 진행상황 확인용입니다. 성공 완료 시 cleanup_checkpoint_on_success=True 때문에 checkpoint는 비워질 수 있습니다.
if "batch_result" in globals():
    display(
        spark.table(PIPELINE_PROGRESS_TABLE)
        .where(F.col("checkpoint_key") == batch_result["checkpoint_key"])
        .orderBy(F.col("created_at").desc())
    )
else:
    display(
        spark.table(PIPELINE_PROGRESS_TABLE)
        .where(F.col("pipeline_name") == "taxonomy_classification_batch")
        .orderBy(F.col("created_at").desc())
        .limit(100)
    )